# Embedding SigLIP-2 cho keyframe

Trích vector ảnh `google/siglip2-so400m-patch14-384` (1152 chiều, chuẩn hoá L2) cho mọi keyframe.

- **Đầu vào:** thư mục keyframe `<video_id>/<n>.jpg` (dataset Kaggle).
- **Đầu ra:** `/kaggle/working/embeddings/siglip/<video_id>.npy`, dòng *i* là ảnh thứ *i* của video.
- **Sau đó:** chép vào `data/embeddings/siglip/` trên máy build rồi chạy
  `python scripts/build_faiss_index.py`.

Bản CLI tương đương để chạy ngoài Kaggle: `scripts/embed_keyframes.py --model siglip2`.
Bật GPU (T4 x2 hoặc P100) trong Settings của notebook.

In [ ]:
!pip install -q "transformers>=4.56" accelerate

## Cấu hình
Chia việc cho nhiều máy: mỗi người đặt `SHARD_INDEX` khác nhau (1..`NUM_SHARDS`).

In [ ]:
from pathlib import Path

KEYFRAME_DIR = Path("/kaggle/input/datasets/trncqun/aic26-keyframes")
MAP_DIR = None  # ví dụ Path("/kaggle/input/aic26-map-keyframes/map-keyframes") để kiểm tra thứ tự
OUTPUT_DIR = Path("/kaggle/working/embeddings/siglip")
ONLY_VIDEOS = []  # rỗng = mọi video; hoặc ["M01", "L21_V001"]

SHARD_INDEX = 1  # ← đổi theo máy: 1, 2, 3 hoặc 4
NUM_SHARDS = 4

MODEL_NAME = "google/siglip2-so400m-patch14-384"
# Hub commit: the backend embeds queries with the same one (AIC_SIGLIP_MODEL_REVISION)
MODEL_REVISION = "e8e487298228002f3d8a82e0cd5c8ea9c567f57f"
EMBEDDING_DIM = 1152
BATCH_SIZE = 64
NUM_WORKERS = 2

## Quét keyframe
Mỗi thư mục có tên dạng video id (`L21_V001`, `M01_V001`, `N001-V001`, ...) ở bất kỳ độ sâu nào.
Ảnh trong video được sắp theo **số** trong tên file (`2.jpg` trước `10.jpg`), đúng thứ tự dòng
`n` của `map-keyframes/<video_id>.csv` và thứ tự mà `build_faiss_index.py` mong đợi.

In [ ]:
import csv
import re
import time

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

VIDEO_ID_RE = re.compile(r"^[A-Z]\d{2,3}[-_]V\d{3}$")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}


def select_shard(items, shard_index, num_shards, weight=None):
    """Part shard_index (1-based) of num_shards. Items are sorted first so every machine agrees.

    Without weight: contiguous chunks. With weight: the heaviest item goes to the lightest
    shard (e.g. by video size), so all machines finish at about the same time.
    """
    assert 1 <= shard_index <= num_shards, "SHARD_INDEX phải nằm trong [1, NUM_SHARDS]"
    ordered = sorted(items)
    if num_shards == 1:
        return ordered
    if weight is None:
        size = -(-len(ordered) // num_shards)
        return ordered[(shard_index - 1) * size : shard_index * size]
    shards = [[] for _ in range(num_shards)]
    totals = [0.0] * num_shards
    for item in sorted(ordered, key=weight, reverse=True):
        lightest = totals.index(min(totals))
        shards[lightest].append(item)
        totals[lightest] += weight(item)
    return sorted(shards[shard_index - 1])


def matches_any(video_id, selectors):
    """True when no selector is given or video_id equals / starts with one of them."""
    return not selectors or any(video_id == s or video_id.startswith(s) for s in selectors)


def scan_keyframes(root):
    """video_id -> images sorted by the number in their file name."""
    videos = {}
    video_dirs = sorted(p for p in root.rglob("*") if p.is_dir() and VIDEO_ID_RE.match(p.name))
    for video_dir in video_dirs:
        images = [
            f for f in video_dir.iterdir() if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
        ]
        if not images:
            continue
        bad = [f.name for f in images if not f.stem.isdigit()]
        if bad:
            raise ValueError(f"{video_dir}: tên ảnh không phải số {bad[:5]}")
        if video_dir.name in videos:
            raise ValueError(f"Trùng video_id {video_dir.name}: {video_dir}")
        videos[video_dir.name] = sorted(images, key=lambda f: int(f.stem))
    return videos


def check_against_map(videos, map_dir):
    """Image numbers must equal the n column of map-keyframes, in the same order."""
    problems = []
    for video_id, images in videos.items():
        map_csv = map_dir / f"{video_id}.csv"
        if not map_csv.exists():
            problems.append(f"{video_id}: không có map CSV")
            continue
        with open(map_csv, encoding="utf-8") as f:
            numbers = [int(row["n"]) for row in csv.DictReader(f)]
        if [int(p.stem) for p in images] != numbers:
            problems.append(f"{video_id}: {len(images)} ảnh, map có {len(numbers)} dòng")
    if problems:
        raise ValueError("Keyframe không khớp map-keyframes:\n" + "\n".join(problems[:20]))
    print(f"[OK] {len(videos)} video khớp map-keyframes (số lượng và thứ tự)")


all_videos = {v: p for v, p in scan_keyframes(KEYFRAME_DIR).items() if matches_any(v, ONLY_VIDEOS)}
videos = {v: all_videos[v] for v in select_shard(list(all_videos), SHARD_INDEX, NUM_SHARDS)}
if MAP_DIR is not None:
    check_against_map(videos, MAP_DIR)
print(
    f"Shard {SHARD_INDEX}/{NUM_SHARDS}: {len(videos)}/{len(all_videos)} video, "
    f"{sum(map(len, videos.values()))} keyframe"
)

## Model
Khung hình 16:9 được **pad thành hình vuông** bằng viền đen trước khi vào processor, nếu không
processor sẽ cắt mất hai bên ảnh. Phải giữ đúng cách này để vector khớp với index đang dùng.

In [ ]:
from transformers import AutoModel, AutoProcessor

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
processor = AutoProcessor.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
model = AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION).to(DEVICE).eval()
assert model.config.vision_config.hidden_size == EMBEDDING_DIM


def preprocess(image):
    width, height = image.size
    side = max(width, height)
    padded = Image.new("RGB", (side, side), (0, 0, 0))
    padded.paste(image, ((side - width) // 2, (side - height) // 2))
    return processor(images=padded, return_tensors="pt")["pixel_values"].squeeze(0)


def encode(pixel_values):
    return model.vision_model(pixel_values=pixel_values).pooler_output


print(f"{MODEL_NAME} trên {DEVICE}")

## Trích embedding
Ảnh lỗi làm **dừng** video đó thay vì bị bỏ qua: bỏ một ảnh sẽ làm mọi dòng phía sau gắn nhầm
keyframe. Video đã có `.npy` đúng kích thước được bỏ qua, nên chạy lại cell này để tiếp tục
sau khi bị ngắt.

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, paths, preprocess):
        self.paths = paths
        self.preprocess = preprocess

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            return self.preprocess(image.convert("RGB"))


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def done(video_id):
    path = OUTPUT_DIR / f"{video_id}.npy"
    if not path.exists():
        return False
    try:
        return np.load(path, mmap_mode="r").shape == (len(videos[video_id]), EMBEDDING_DIM)
    except ValueError:
        return False


pending = [v for v in videos if not done(v)]
failed = {}
print(f"Đã xong {len(videos) - len(pending)}, còn {len(pending)} video")
start = time.time()
with torch.inference_mode(), torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
    for video_id in tqdm(pending, desc="Video"):
        loader = DataLoader(
            ImageDataset(videos[video_id], preprocess),
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS,
            pin_memory=DEVICE.type == "cuda",
        )
        try:
            chunks = []
            for pixel_values in loader:
                embeddings = encode(pixel_values.to(DEVICE, non_blocking=True))
                chunks.append(
                    torch.nn.functional.normalize(embeddings.float(), dim=-1).cpu().numpy()
                )
        except OSError as exc:
            failed[video_id] = str(exc)
            continue
        matrix = np.vstack(chunks).astype(np.float32)
        assert matrix.shape == (len(videos[video_id]), EMBEDDING_DIM), (video_id, matrix.shape)
        tmp = OUTPUT_DIR / f"{video_id}.npy.tmp"
        with open(tmp, "wb") as f:
            np.save(f, matrix)
        tmp.replace(OUTPUT_DIR / f"{video_id}.npy")

missing = [v for v in videos if not done(v)]
minutes = (time.time() - start) / 60
print(f"Xong {len(videos) - len(missing)}/{len(videos)} video trong {minutes:.1f} phút")
for video_id, error in failed.items():
    print(f"[LỖI] {video_id}: {error}")

## Nén kết quả
Tải file nén ở tab **Output** của Kaggle rồi giải nén vào `data/embeddings/siglip/` trên máy build.

In [ ]:
import shutil

archive = shutil.make_archive(
    str(OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}_shard{SHARD_INDEX}of{NUM_SHARDS}"),
    "zip",
    root_dir=str(OUTPUT_DIR.parent),
    base_dir=OUTPUT_DIR.name,
)
print(f"{archive}: {Path(archive).stat().st_size / 2**20:.1f} MB")